# Play 1 — Agent Instructions Generator

> **Part of the Tableau + Microsoft Fabric AI Bridge project.**

This utility notebook auto-generates the agent instructions for the Azure AI Foundry agent
by reading field metadata directly from a Tableau published datasource via the VizQL Data
Service API.

**No Fabric lakehouse required.** All it needs is a Tableau PAT in Key Vault and the
datasource display name — the LUID is resolved automatically.

**How to use:**
1. Fill in Cell 1 with your Key Vault details, Tableau credentials, and datasource name
2. Run all cells
3. Copy the output from Cell 5
4. Paste it into the **Instructions** field when creating your Azure AI Foundry agent


## ⚠️ Start Here — Plug In Your Variables

| Variable | What it is | Where to find it |
|----------|-----------|------------------|
| `PAT_NAME` | Tableau PAT name | Tableau → Account Settings → Personal Access Tokens |
| `KV_URL` | Azure Key Vault URL | `https://<your-vault-name>.vault.azure.net/` |
| `KV_SECRET_NAME` | Secret name in Key Vault | The name you gave the secret storing your PAT |
| `POD` | Tableau Cloud pod hostname | First part of your Tableau Cloud URL |
| `SITE` | Site contentUrl slug | Your site URL slug. Use `""` for default site |
| `DATASOURCE_NAME` | Exact datasource display name | As it appears in Tableau — case sensitive |


## Cell 1 — Configuration

Set your Tableau credentials and datasource LUID here.

In [ ]:
# ── TABLEAU CONNECTION ────────────────────────────────────────────────────────
PAT_NAME         = ""   # PAT name from Tableau account settings
POD              = ""   # e.g. 10ay.online.tableau.com
SITE             = ""   # Site contentUrl slug. Use "" for default site

# ── KEY VAULT ─────────────────────────────────────────────────────────────────
KV_URL           = "https://<your-keyvault-name>.vault.azure.net/"
KV_SECRET_NAME   = "<your-secret-name>"
PAT_SECRET       = notebookutils.credentials.getSecret(KV_URL, KV_SECRET_NAME)

# ── DATASOURCE ────────────────────────────────────────────────────────────────
DATASOURCE_NAME  = ""   # Exact display name of your datasource e.g. "Superstore Datasource"

BASE = f"https://{POD}"

print("✓ Configuration loaded")
print(f"  Pod:              {POD}")
print(f"  Site:             {SITE or '(default)'}")
print(f"  Datasource name:  {DATASOURCE_NAME}")
print(f"  PAT secret:       retrieved from Key Vault ✓")


## Cell 2 — Authenticate to Tableau

In [ ]:
import requests

auth_response = requests.post(
    f"{BASE}/api/3.24/auth/signin",
    json={
        "credentials": {
            "personalAccessTokenName": PAT_NAME,
            "personalAccessTokenSecret": PAT_SECRET,
            "site": {"contentUrl": SITE}
        }
    },
    headers={"Content-Type": "application/json", "Accept": "application/json"}
)
auth_response.raise_for_status()

auth_data = auth_response.json()
TOKEN   = auth_data["credentials"]["token"]
SITE_ID = auth_data["credentials"]["site"]["id"]

HEADERS = {
    "X-Tableau-Auth": TOKEN,
    "Content-Type": "application/json",
    "Accept": "application/json"
}

print("✓ Authenticated to Tableau")
print(f"  Token:    {TOKEN[:8]}...")
print(f"  Site ID:  {SITE_ID}")


## Cell 3 — Resolve Datasource LUID

Looks up the datasource GUID from the Tableau REST API using the display name you provided in Cell 1. If the name doesn't match exactly, the cell will print all available datasources so you can find the right one.

In [ ]:
# Resolve datasource LUID from display name via REST API
resp = requests.get(
    f"{BASE}/api/3.24/sites/{SITE_ID}/datasources",
    headers=HEADERS
)
resp.raise_for_status()
datasources = resp.json().get("datasources", {}).get("datasource", [])
if isinstance(datasources, dict):
    datasources = [datasources]

match = next((ds for ds in datasources if ds["name"].lower() == DATASOURCE_NAME.lower()), None)

if match:
    DATASOURCE_LUID = match["id"]
    print(f"✓ Datasource found")
    print(f"  Name: {match['name']}")
    print(f"  LUID: {DATASOURCE_LUID}")
else:
    print(f"✗ Datasource '{DATASOURCE_NAME}' not found")
    print("\nAvailable datasources on this site:")
    for ds in datasources:
        print(f"  {ds['name']}: {ds['id']}")
    raise ValueError(f"Datasource '{DATASOURCE_NAME}' not found — check the name above and try again")


## Cell 4 — Fetch Field Metadata from VDS

In [ ]:
# Fetch field metadata from VDS
resp = requests.post(
    f"{BASE}/api/v1/vizql-data-service/read-metadata",
    json={"datasource": {"datasourceLuid": DATASOURCE_LUID}},
    headers=HEADERS
)
resp.raise_for_status()
all_fields = resp.json().get("data", [])

# Separate dimensions and measures — skip bins, sets, groups, calculated fields
dimensions = []
measures   = []

for f in all_fields:
    if f.get("columnClass") != "COLUMN":
        continue
    caption = f.get("fieldCaption", "")
    role    = f.get("fieldRole", "")
    if not caption:
        continue
    if role == "MEASURE":
        measures.append(caption)
    else:
        dimensions.append(caption)

print(f"✓ Fields retrieved from VDS")
print(f"  Dimensions: {len(dimensions)}")
print(f"  Measures:   {len(measures)}")
print(f"\n  Dimensions: {dimensions}")
print(f"  Measures:   {measures}")


## Cell 5 — Generate Agent Instructions

Run this cell and copy everything between the dividers into your Foundry agent Instructions field.

In [ ]:
# Format the field list for agent instructions
dim_list     = "\n".join([f"- {d}" for d in dimensions])
measure_list = "\n".join([f"- {m} (use SUM, AVG, MIN, MAX, or MEDIAN)" for m in measures])

instructions = f"""# Foundry Agent Instructions — {DATASOURCE_NAME}

> Paste this into the **Instructions** field when creating the Azure AI Foundry agent.

---

You are a data analyst agent with direct access to the {DATASOURCE_NAME} Tableau data source
via the VizQL Data Service API. You can query live Tableau data to answer business questions
in natural language.

You have access to the following fields:

**Dimensions** (use as-is, no aggregation needed):
{dim_list}

**Measures** (always aggregate — never request raw):
{measure_list}

When a user asks a data question:
1. Call queryTableauData with a well-constructed query_fields array containing only the
   fields needed to answer the question
2. Synthesize the returned data into a clear, concise natural language answer
3. Include specific numbers and rankings where relevant

QUERY CONSTRUCTION RULES:
- Always aggregate measures — never request a measure field without a function (SUM, AVG, MIN, MAX, MEDIAN)
- Always apply a date function to date fields — never request them as raw dates.
  Use YEAR for annual analysis, QUARTER or MONTH for trend analysis
- To count unique records, use a dimension field with function COUNTD
- Dimensions do not need a function
- Only request fields necessary to answer the question — keep payloads small
- Avoid high cardinality dimensions as standalone fields — they will produce oversized results

If the user asks something that can't be answered from this dataset, say so clearly.

---

## Notes for replication

- Authentication is handled internally by the Logic App — do not add a separate auth tool
- The queryTableauData tool is defined in the OpenAPI spec (openapi_spec.json)
- This field list was auto-generated from VDS read-metadata on {DATASOURCE_NAME}
"""

print("=" * 60)
print("DATASOURCE LUID — USE THIS IN YOUR DEPLOY COMMAND")
print("=" * 60)
print(f"  Datasource:           {DATASOURCE_NAME}")
print(f"  tableau_datasource_luid: {DATASOURCE_LUID}")
print("  → Copy this GUID into the deploy command:")
print(f"    tableau_datasource_luid={DATASOURCE_LUID}")
print("=" * 60)
print()
print("=" * 60)
print("COPY EVERYTHING BELOW THIS LINE INTO FOUNDRY AGENT INSTRUCTIONS")
print("=" * 60)
print()
print(instructions)
print("=" * 60)
print("END OF AGENT INSTRUCTIONS")
print("=" * 60)
